# Email Finder — Part 1: Discovery + Mailin Submit (EF-19)

Pipeline:
1. Define leads inline
2. Qualify existing emails
3. Run discovery waterfall (Perplexity → scraper → pattern gen)
4. Inspect Perplexity responses
5. Submit candidates to Mailin → save task state JSON

**Run Part 2 (`mailin_results.ipynb`) after Mailin finishes processing.**

## Cell 1: Setup & Config

In [ ]:
import sys, io, json, textwrap, logging, datetime
import pandas as pd
sys.path.append("../..")

from email_finder import LeadInput, EmailFinderResult, load_leads_from_csv, load_leads_from_google_sheet
from email_finder.config import Config
from email_finder.io.loader import _row_to_lead
from email_finder.io.qualify import qualify_lead_email, is_email_qualified
from email_finder.batch import discover_leads, collect_candidates
from email_finder.verification.mailin_automator import mailin_submit_batch

logging.basicConfig(level=logging.INFO)
config = Config()
print("Config loaded.")

## Cell 2: Define Test Leads (inline CSV)

`Podcast Name` → `full_name`.  
Perplexity finds the host via podcast name + website.

In [ ]:
RAW_CSV = textwrap.dedent("""\
Podcast Name,Podcast Website,Podcast Email,Podcast Facebook,Podcast Twitter,Podcast Instagram,Podcast YouTube,Podcast LinkedIn
A Mommy And A Mic,http://www.amommyandamic.com/,podcast@myrockerbeez.com,,,,,
Culinary Treasure Podcast,https://www.culinarytreasurepodcast.com/,sshomler@me.com,https://www.facebook.com/CulinaryTreasurePodcast,,https://www.instagram.com/culinarytreasurepodcast,https://www.youtube.com/channel/UCA-zUuYpU_KQpVemaheeWEA,
Joy of Weightlessness,https://www.buzzsprout.com/2016334,,https://www.facebook.com/joalibng,https://twitter.com/JOALIBEING,https://www.instagram.com/joalibeing,,
TravelRight.Today,http://www.travelright.today/,,,,,,
Paper Trails,https://papertrails.podbean.com/,,,,,,
Waves of Impact,https://uwf.edu/commerce,,,,,,
The Smoking Barrel Podcast,http://thesmokingbarrelpodcast.com/,,https://www.facebook.com/thesmokingbarrelpodcast,,,,
The Southern Fork,http://www.thesouthernfork.com/episodes/,charlotteghost@gmail.com,,.,"https://www.instagram.com/southernfork",,
Chef D's Bistro,https://podcasters.spotify.com/pod/show/darryl-ingram,darryl.ingram0162@gmail.com,,,,,
Grounded,https://www.groundedthepod.com/,"smoody09@gmail.com, tech@ringmaster.com",https://www.facebook.com/MichaelKLaRue,,https://www.instagram.com/tridavetri,,https://www.linkedin.com/in/amyhom17
""")

COLUMN_MAPPING = {
    "Podcast Name":      "full_name",
    "Podcast Website":   "website",
    "Podcast Email":     "existing_email",
    "Podcast Facebook":  "facebook_url",
    "Podcast Twitter":   "twitter_url",
    "Podcast Instagram": "instagram_url",
    "Podcast YouTube":   "youtube_url",
    "Podcast LinkedIn":  "linkedin_url",
}

df = pd.read_csv(io.StringIO(RAW_CSV), dtype=str, keep_default_na=False).replace("nan", "")
print(f"Parsed {len(df)} rows")
df

## Cell 3: Build LeadInput Objects

In [ ]:
raw_leads = []
for _, row in df.iterrows():
    lead = _row_to_lead(row, COLUMN_MAPPING)
    if lead:
        raw_leads.append(lead)

print(f"Built {len(raw_leads)} LeadInput objects:")
for l in raw_leads:
    print(f"  {l.full_name:<35}  email={l.existing_email or '—'}")

## Cell 4: Qualify Existing Emails

- Hosting platform domain → DROP
- Personal provider (gmail, me.com …) → KEEP
- Custom domain with LCS < 4 vs brand candidates → DROP → Flow B

In [ ]:
leads = []
for lead in raw_leads:
    qualified_lead, reason = qualify_lead_email(lead)
    leads.append(qualified_lead)

    if reason:
        print(f"  [DISCARD] {lead.full_name}")
        print(f"            email: {lead.existing_email}")
        print(f"            reason: {reason}")
    elif lead.existing_email:
        print(f"  [KEEP]    {lead.full_name}  →  {lead.existing_email}")
    else:
        print(f"  [NO EMAIL] {lead.full_name} → Flow B")

print(f"\n{sum(1 for l in leads if l.existing_email)} leads with qualified email (Flow A)")
print(f"{sum(1 for l in leads if not l.existing_email)} leads with no email (Flow B)")

## Cell 5: Run Discovery Waterfall

Runs Perplexity → website scraper → pattern generator per lead.  
Does **not** call Mailin yet.

In [ ]:
pre_results = await discover_leads(leads, config)

## Cell 6: Inspect Perplexity Responses

Review what Perplexity returned before submitting to Mailin.

In [ ]:
for lead, result in zip(leads, pre_results):
    perplexity_entries = [e for e in result.discovery_log if e.get("node") == "perplexity"]
    if not perplexity_entries:
        continue

    print(f"\n{'='*80}")
    print(f"LEAD: {lead.full_name}")

    for i, entry in enumerate(perplexity_entries, 1):
        res = entry.get("result", {})
        print(f"  [Perplexity call #{i}]")
        print(f"  PROMPT:")
        print(f"    {res.get('prompt', '—')}")
        print(f"  FOUND EMAIL: {res.get('found_email') or res.get('found_emails') or '—'}")
        print(f"  RAW RESPONSE:")
        raw = res.get('raw_response', '')
        print(f"    {raw[:600]}{'...' if len(raw) > 600 else ''}")

## Cell 7: Submit to Mailin

Uploads all candidate emails and saves the Task ID to a JSON file.  
**Mailin processes this asynchronously** — check the Mailin dashboard and run `mailin_results.ipynb` when status changes from *Verifying* to *Completed*.

In [ ]:
import os
os.makedirs("./output", exist_ok=True)

all_candidates = collect_candidates(pre_results)
print(f"Submitting {len(all_candidates)} candidate email(s) to Mailin …")

task_id = await mailin_submit_batch(all_candidates, config, headless=True)
print(f"\n✓ Submitted — Task ID: {task_id}")
print(f"  Check https://app.mailin.ai/verification (Task Results tab) for status.")

## Cell 8: Save Task State

In [ ]:
STATE_PATH = "./output/mailin_task_state.json"

task_state = {
    "task_id": task_id,
    "submitted_at": datetime.datetime.utcnow().isoformat(),
    "all_candidates": all_candidates,
    "leads": [l.model_dump() for l in leads],
    "pre_results": [
        {
            "email": r.email,
            "status": r.status,
            "confidence": r.confidence,
            "source": r.source,
            "discovery_log": r.discovery_log,
            "verification_details": r.verification_details,
        }
        for r in pre_results
    ],
}

with open(STATE_PATH, "w") as f:
    json.dump(task_state, f, indent=2)

print(f"Task state saved to {STATE_PATH}")
print(f"Task ID : {task_id}")
print(f"Leads   : {len(leads)}")
print(f"Emails  : {len(all_candidates)}")
print(f"\nNow open mailin_results.ipynb once Mailin shows status = Completed.")

## Cell 9: Quick Email Qualification Tester

Test qualification logic against any email + podcast name.

In [ ]:
test_cases = [
    ("podcast@myrockerbeez.com",      "A Mommy And A Mic",          "http://www.amommyandamic.com/"),
    ("sshomler@me.com",               "Culinary Treasure Podcast",   "https://www.culinarytreasurepodcast.com/"),
    ("charlotteghost@gmail.com",      "The Southern Fork",           "http://www.thesouthernfork.com/"),
    ("smoody09@gmail.com",            "Grounded",                    "https://www.groundedthepod.com/"),
    ("darryl.ingram0162@gmail.com",   "Chef D's Bistro",             "https://podcasters.spotify.com/pod/show/darryl-ingram"),
]

print(f"{'Email':<40}  {'Podcast':<30}  {'Result':<8}  Reason")
print("-" * 110)
for email, name, site in test_cases:
    ok, reason = is_email_qualified(email, name, site)
    icon = "✓ KEEP" if ok else "✗ DROP"
    print(f"{email:<40}  {name:<30}  {icon:<8}  {reason}")